# `get_results.ipynb` — analysis of the kriging outputs

This notebook reads the rasters and metrics produced by the `kriging_*.py` scripts in this
folder and turns them into the paper's numbers and figures. It does **not** run any kriging
itself — run `kriging.sh` first (see `README.md`).

Every input path is resolved through **`sumatra.paths`**, the same module the scripts use to
*write* their outputs, so the notebook can never read from a stale/renamed file again. All the
data it needs is bundled in **`sumatra/data/`**, so a fresh clone runs this end to end with no
external data and no `AGBD_ENV` — it works from any cwd.

## What each section does

1. **Agreement between GEDI L4B 1km and Sumatra 1km** — sanity check that our 1 km AGB map
   agrees with the independent GEDI L4B product.
2. **`<map>, post-kriging on GEDI`** — one section per corrected map (*our* 10 m map, *our*
   downsampled map, the *ESA CCI* map). Each follows the **same four steps**, which are the
   shared helpers defined in the *Shared helpers* cell:
     1. `load_pre_post_ref` — load the pre-kriging map, its post-kriging correction, and the reference,
     2. `print_metrics` — RMSE / MAE / ME / R² pre vs. post,
     3. `show_maps` (+ `save_map_panels`) — full and zoomed AGB maps,
     4. `show_binned_residuals` and `show_density` (+ `save_scatter_panels`) — error vs. AGB.
3. **Agreement of ours and CCI vs. GEDI (footprint-level)** — evaluate at the held-out GEDI
   test footprints rather than per pixel.

The `save_*_panels` helpers write the individual PNGs that `compose_figure.py` assembles into
the final figures.

> **Deprecated:** the two *post-kriging on field* sections below are kept for history only — the
> field/Mozambique reference path was removed from the pipeline, so those cells no longer run
> (their code cells are marked *raw*).

In [ ]:
# `paths` is the SAME module the kriging_*.py scripts write through, so notebook inputs are
# guaranteed to match script outputs. It reads from the bundled sumatra/data/ when present and
# falls back to the full DATA_ROOT tree otherwise, so no AGBD_ENV is required just to run this.
from os.path import join
from sumatra import paths

# The run whose outputs we analyse (matches kriging.sh defaults).
ARCH, YEAR = 'nico_film', 2020
ENS = ['17997535-1', '17997535-2', '17997535-3']
ENS_STR = '_'.join(ENS)
REFERENCE, STRIPE = 'gedi', 50

# Uncorrected input maps that were kriged (band 1 = AGB).
MERGED_INPUT = paths.merged_input(ENS)   # our downsampled 100 m prediction
CCI_INPUT    = paths.cci_input()         # ESA CCI map

# Zoom window (row0, row1, col0, col1) used for the zoomed map panels.
ZOOM = (500, 1000, 550, 1050)

In [ ]:
import rasterio as rs
import numpy as np
import matplotlib.pyplot as plt
from rasterio.features import shapes
from shapely.geometry import shape
from matplotlib.colors import ListedColormap
import pickle
import os
from os.path import isdir, join, isfile
from rasterio.enums import Resampling
from matplotlib.colors import LogNorm
from sklearn.metrics import r2_score

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Shared helpers — the four steps every "<map>, post-kriging on GEDI" section repeats.
# Defined once here so the sections stay short and behave identically; if you prefer, they
# can be moved verbatim into a `sumatra/results_plots.py` module and imported.
# Corrected GeoTIFFs have bands: (1) AGB, (2) Residuals, (3) STD.
# ─────────────────────────────────────────────────────────────────────────────

def load_pre_post_ref(pre_path, post_path, resolution=100, resample_pre=True, std_band=None):
    """Load a pre-kriging map, its post-kriging correction, and the GEDI reference raster,
    all as 2D arrays on the reference grid. If `std_band` is given, also return that band of
    the post map. `resample_pre` bilinearly resamples the pre map onto the reference grid."""
    with rs.open(paths.reference_agb(resolution)) as src:
        ref = src.read(1)
    with rs.open(pre_path) as src:
        if resample_pre:
            pre = src.read(1, out_shape=(1, ref.shape[0], ref.shape[1]), resampling=Resampling.bilinear)
        else:
            pre = src.read(1)
    with rs.open(post_path) as src:
        post = src.read(1)
        std = src.read(std_band) if std_band else None
    return (pre, post, ref, std) if std_band else (pre, post, ref)


def print_metrics(pre, post, ref, with_r2=True):
    """Print pre/post RMSE, MAE, ME (+ R2) over pixels where ref/pre/post are all valid.
    Returns (pre_residuals, post_residuals, valid_mask) for the plotting helpers."""
    valid = (~np.isnan(ref)) & (~np.isnan(pre)) & (~np.isnan(post))
    pre_res, post_res = pre[valid] - ref[valid], post[valid] - ref[valid]
    print(f"{'Metric':<10} {'Pre-Kriging':<15} {'Post-Kriging':<15}")
    print('-' * 40)
    print(f"{'RMSE':<10} {np.sqrt(np.nanmean(pre_res**2)):<15.2f} {np.sqrt(np.nanmean(post_res**2)):<15.2f}")
    print(f"{'MAE':<10} {np.nanmean(np.abs(pre_res)):<15.2f} {np.nanmean(np.abs(post_res)):<15.2f}")
    print(f"{'ME':<10} {np.nanmean(pre_res):<15.2f} {np.nanmean(post_res):<15.2f}")
    if with_r2:
        print(f"{'R2':<10} {r2_score(ref[valid], pre[valid]):<15.4f} {r2_score(ref[valid], post[valid]):<15.4f}")
    return pre_res, post_res, valid


def show_maps(pre, post, ref, save=None, zoom=None, vmin=0, vmax=250):
    """3-panel pre / post / reference AGB maps (optionally cropped to a (r0,r1,c0,c1) zoom)."""
    if zoom is not None:
        r0, r1, c0, c1 = zoom
        pre, post, ref = pre[r0:r1, c0:c1], post[r0:r1, c0:c1], ref[r0:r1, c0:c1]
    f, axarr = plt.subplots(1, 3, figsize=(15, 15), constrained_layout=True)
    titles = ('Pre-Kriging AGB Density [t/ha]', 'Post-Kriging AGB Density [t/ha]', 'Reference AGB Density [t/ha]')
    for ax, data, title in zip(axarr, (pre, post, ref), titles):
        im = ax.imshow(data, cmap='viridis', vmin=vmin, vmax=vmax)
        ax.set_title(title); ax.axis('off')
    f.colorbar(im, ax=axarr, shrink=0.2, orientation='vertical').set_label('AGB Density [t/ha]')
    if save: plt.savefig(save, dpi=1200, bbox_inches='tight')
    plt.show()


def show_post_std(post, std, vmax_std=50):
    """2-panel post-kriging AGB map + its kriging standard deviation."""
    f, axarr = plt.subplots(1, 2, figsize=(15, 15), constrained_layout=True)
    im1 = axarr[0].imshow(post, cmap='viridis', vmin=0, vmax=250)
    axarr[0].set_title('Post-Kriging AGB Density [t/ha]'); axarr[0].axis('off')
    f.colorbar(im1, ax=axarr[0], shrink=0.6, orientation='vertical').set_label('AGB Density [t/ha]')
    im2 = axarr[1].imshow(std, cmap='viridis', vmin=0, vmax=vmax_std)
    axarr[1].set_title('Post-Kriging Standard Deviation [t/ha]'); axarr[1].axis('off')
    f.colorbar(im2, ax=axarr[1], shrink=0.6, orientation='vertical').set_label('AGB Standard Deviation [t/ha]')
    plt.show()


# Colour-blind-friendly palette: (facecolor, edgecolor, mediancolor) per box series.
_BLUE  = ('#56B4E9', '#0072B2', '#004E7C')
_GREEN = ('#5EC8A5', '#009E73', '#006B4F')
_AMBER = ('#F5C862', '#E69F00', '#B37A00')

def show_binned_residuals(ref_test, series, save=None, show_ylabel=True,
                          bins=np.arange(0, 300, 50)):
    """Box plots of residuals binned by reference AGB. `series` is a list of
    (residuals, (face, edge, median), label) tuples — one box group per entry."""
    lbs, ubs = bins[:-1], bins[1:]
    labels = [f"{lb}-{ub}" for lb, ub in zip(lbs, ubs)]
    n = len(series)
    width, span = (10, (10, 30)) if n >= 3 else (10, (10, 20)) if n == 2 else (20, (20, 20))
    offsets = np.linspace(span[0], span[1], n)
    fig, ax = plt.subplots(figsize=(10, 6))
    boxes, legend = [], []
    for (res, (face, edge, med), lab), off in zip(series, offsets):
        binned = [res[(ref_test >= lb) & (ref_test < ub)] for lb, ub in zip(lbs, ubs)]
        bx = ax.boxplot(binned, showfliers=False, positions=bins[:-1] + off, widths=width, patch_artist=True,
                        boxprops=dict(facecolor=face, color=edge), medianprops=dict(color=med),
                        whiskerprops=dict(color=edge), capprops=dict(color=edge),
                        flierprops=dict(color=edge, markeredgecolor=edge))
        boxes.append(bx["boxes"][0]); legend.append(lab)
    ax.axhline(0, color='grey', linestyle='--', alpha=0.7, linewidth=1)
    ax.set_xticks(bins[:-1] + 20); ax.set_xticklabels(labels)
    ax.set_ylim(-250, 300)
    ax.set_xlabel('AGB Density Bins [t/ha]')
    ax.set_ylabel('Residuals [t/ha]' if show_ylabel else '')
    if not show_ylabel: ax.set_yticklabels([])
    ax.legend(boxes, legend, loc='upper left')
    if save: plt.savefig(save, dpi=1200)
    plt.show()


def show_density(pre, post, ref, save=None, ma=350, vmax_count=10000):
    """Side-by-side density scatter (predicted vs. reference AGB), pre and post kriging."""
    fig, axs = plt.subplots(1, 2, figsize=(18, 8))
    ticks = np.arange(0, ma + 50, 50)
    for ax, pred in zip(axs, (pre, post)):
        mask = np.isnan(ref) | np.isnan(pred)
        h = ax.hist2d(ref[~mask].squeeze(), pred[~mask].squeeze(), bins=(100, 100),
                      cmap='Greens', norm=LogNorm(vmin=1, vmax=vmax_count))
        cbar = fig.colorbar(h[3], ax=ax); cbar.ax.tick_params(labelsize=14)
        cbar.ax.set_ylabel('Number of samples', fontsize=16)
        ax.plot([0, ma], [0, ma], 'k--'); ax.set_xlim(0, ma); ax.set_ylim(0, ma)
        ax.set_xlabel('GEDI reference AGBD [Mg/ha]', fontsize=16)
        ax.set_ylabel('Predicted AGB [Mg/ha]', fontsize=16)
        ax.set_xticks(ticks); ax.set_yticks(ticks); ax.tick_params(labelsize=14)
        ax.grid(); ax.set_aspect('equal')
    plt.tight_layout()
    if save: plt.savefig(save, dpi=1200)
    plt.show()


def save_map_panels(panels, vmin=0, vmax=250, cmap='viridis', dpi=300):
    """Save each named AGB-map panel as a standalone transparent PNG (for compose_figure.py)."""
    for path, data in panels.items():
        fig, ax = plt.subplots(figsize=(5, 5), constrained_layout=True)
        ax.imshow(data, cmap=cmap, vmin=vmin, vmax=vmax); ax.axis('off')
        fig.savefig(path, dpi=dpi, bbox_inches='tight', pad_inches=0, transparent=True)
        plt.close(fig); print(f'Saved {path}')


def save_scatter_panels(panels, ma=350, cmap='Greens', vmin=1, vmax=10000, dpi=300):
    """Save each named density-scatter panel as a standalone PNG (for compose_figure.py).
    `panels` maps path -> (reference_array, prediction_array)."""
    ticks = np.arange(0, ma + 50, 50)
    for path, (ref, pred) in panels.items():
        fig, ax = plt.subplots(figsize=(6, 6))
        mask = np.isnan(ref) | np.isnan(pred)
        ax.hist2d(ref[~mask].squeeze(), pred[~mask].squeeze(), bins=(100, 100),
                  cmap=cmap, norm=LogNorm(vmin=vmin, vmax=vmax))
        ax.plot([0, ma], [0, ma], 'k--'); ax.set_xlim(0, ma); ax.set_ylim(0, ma)
        ax.set_xticks(ticks); ax.set_yticks(ticks); ax.tick_params(labelsize=14)
        ax.grid(); ax.set_aspect('equal')
        fig.savefig(path, dpi=dpi, bbox_inches='tight', pad_inches=0.05)
        plt.close(fig); print(f'Saved {path}')

## Agreement between GEDI L4B 1km and Sumatra 1km

In [ ]:
with rs.open(paths.reference_agb(1000), 'r') as src:
    sumatra_1 = src.read(1)
    target_crs = src.crs

with rs.open(paths.gedi_l4b(), 'r') as src:
    gedi_1km = src.read(1)
    gedi_transform = src.transform
    gedi_crs = src.crs

# Reproject GEDI to Sumatra CRS
from rasterio.warp import calculate_default_transform, reproject, Resampling
from rasterio.enums import Resampling
gedi_1km_reprojected = np.full(sumatra_1.shape, np.nan, dtype=np.float32)
transform, width, height = calculate_default_transform(
    gedi_crs, target_crs, src.width, src.height, *src.bounds)
reproject(
    source=gedi_1km,
    destination=gedi_1km_reprojected,
    src_transform=gedi_transform,
    src_crs=gedi_crs,
    dst_transform=transform,
    dst_crs=target_crs,
    resampling=Resampling.average)

print(gedi_1km.shape, gedi_1km_reprojected.shape, sumatra_1.shape)

In [ ]:
plt.imshow(gedi_1km, vmin=0.1, vmax=250, cmap='viridis')
plt.colorbar()
plt.show()
plt.imshow(gedi_1km_reprojected, vmin=0.1, vmax=250, cmap='viridis')
plt.colorbar()
plt.show()
plt.imshow(sumatra_1, vmin=0.1, vmax=250, cmap='viridis')
plt.colorbar()
plt.show()

In [ ]:
valid = ~np.isnan(sumatra_1) & (sumatra_1 > 0) & ~np.isnan(gedi_1km_reprojected) & (gedi_1km_reprojected > 0)

sumatra_values = sumatra_1[valid]
gedi_values = gedi_1km_reprojected[valid]

# Compute correlation metrics
from scipy.stats import pearsonr, spearmanr
pearson_corr, _ = pearsonr(gedi_values, sumatra_values)
spearman_corr, _ = spearmanr(gedi_values, sumatra_values)
print(f'Pearson correlation: {pearson_corr:.4f}')
print(f'Spearman correlation: {spearman_corr:.4f}')

# R2 score
from sklearn.metrics import r2_score
r2 = r2_score(gedi_values, sumatra_values)
print(f'R2 score: {r2:.4f}')

print(f'Number of valid pixels: {len(sumatra_values)}')

# density plot
plt.figure(figsize=(8,6))
plt.hist2d(sumatra_values, gedi_values, norm=LogNorm(), bins=100, cmap='viridis')
plt.colorbar()
plt.xlabel('Sumatra AGBD 1km (Mg/ha)')
plt.ylabel('GEDI L4B AGBD 1km (Mg/ha)')
plt.xlim(0, 250)
plt.ylim(0, 250)
plt.plot([0, 250], [0, 250], color='red', linestyle='--')

# Add a box with the correlation metrics
textstr = '\n'.join((
    f'Pearson: {pearson_corr:.3f}',
    f'Spearman: {spearman_corr:.3f}',
    f'$R^2$: {r2:.3f}',
))
props = dict(boxstyle='round', facecolor='white', alpha=0.8)
plt.text(0.73, 0.15, textstr, transform=plt.gca().transAxes, fontsize=10, verticalalignment='top', bbox=props)
plt.savefig('figs/agreement-gedi-sumatra-1km-density.png', dpi=800)
plt.show()

# Scatter plot
plt.figure(figsize=(8,6))
plt.scatter(sumatra_values, gedi_values, alpha=0.5, s=1)
plt.xlabel('Sumatra AGBD 1km (Mg/ha)')
plt.ylabel('GEDI L4B AGBD 1km (Mg/ha)')
plt.title('Scatter Plot of Sumatra AGBD vs GEDI L4B')
plt.xlim(0, 250)
plt.ylim(0, 250)
plt.plot([0, 250], [0, 250], color='red', linestyle='--')
plt.show()



## ~~Post-kriging on field (DEPRECATED)~~

> The field / Mozambique reference correction was **removed** from the pipeline, and the
> outputs these cells read (`predictions/{n}-{stripe}/corr_merged.tif`, `splits/split_map_s=…`)
> are no longer produced. The two sections below are kept **for history only** — their code
> cells are marked *raw* so a full "Run all" skips them. See the GEDI sections further down for
> the current analysis.

### Metrics & plots — binned residuals and density scatter (pre vs. post kriging)

## For the CCI map, post-kriging on field

### Plot the CCI maps (pre- and post-kriging)

### Metrics & plots — binned residuals and density scatter (pre vs. post kriging)

## Our (10 m) prediction, post-kriging on GEDI

In [ ]:
# Our full-resolution prediction, corrected against GEDI.
OURS_MODEL = 'sumatra_gedi_composite'
post_path = paths.corrected_tif(ARCH, YEAR, ENS, OURS_MODEL, REFERENCE, STRIPE)

pre_pred, post_pred, sumatra_1, post_std = load_pre_post_ref(MERGED_INPUT, post_path, std_band=3)
post_std[np.isnan(post_pred)] = np.nan

pre_res, post_res, test_mask = print_metrics(pre_pred, post_pred, sumatra_1)
ours_post_pred_10m = post_pred.copy()  # reused by the downsampled section's 10 m-vs-100 m comparison

In [ ]:
r0, r1, c0, c1 = ZOOM
save_map_panels({
    "figs/ours_basemap.png":         pre_pred,
    "figs/ours_basemap-zoomed.png":  pre_pred[r0:r1, c0:c1],
    "figs/ours_post10m.png":         post_pred,
    "figs/ours_post10m-zoomed.png":  post_pred[r0:r1, c0:c1],
    "figs/reference.png":            sumatra_1,
    "figs/reference-zoomed.png":     sumatra_1[r0:r1, c0:c1],
})

In [ ]:
show_maps(pre_pred, post_pred, sumatra_1, save='figs/ours-gedi-field.png')
show_maps(pre_pred, post_pred, sumatra_1, save='figs/ours-gedi-field-zoom.png', zoom=ZOOM)
show_post_std(post_pred, post_std)

In [ ]:
show_binned_residuals(sumatra_1[test_mask],
    [(pre_res, _BLUE, 'Pre-Kriging'), (post_res, _GREEN, 'Post-Kriging')],
    save='figs/ours-gedi-field-binned-residuals.png')

In [ ]:
show_density(pre_pred, post_pred, sumatra_1, save='figs/ours-gedi-field-density.png')
save_scatter_panels({
    "figs/scatter_ours_basemap.png": (sumatra_1, pre_pred),
    "figs/scatter_ours_10m.png":     (sumatra_1, post_pred),
})

## Our downsampled prediction, post-kriging on GEDI

In [ ]:
# Our downsampled merged prediction, corrected against GEDI.
DOWN_MODEL = 'sumatra_downsampled_gedi_composite'
post_path = paths.corrected_tif(ARCH, YEAR, ENS, DOWN_MODEL, REFERENCE, STRIPE)

pre_pred, post_pred, sumatra_1 = load_pre_post_ref(MERGED_INPUT, post_path)
pre_res, post_res, test_mask = print_metrics(pre_pred, post_pred, sumatra_1, with_r2=False)

In [ ]:
r0, r1, c0, c1 = ZOOM
save_map_panels({
    "figs/ours_post100m.png":        post_pred,
    "figs/ours_post100m-zoomed.png": post_pred[r0:r1, c0:c1],
})

In [ ]:
show_maps(pre_pred, post_pred, sumatra_1, save='figs/ours-downsampled-gedi-field.png')
show_maps(pre_pred, post_pred, sumatra_1, save='figs/ours-downsampled-gedi-field-zoom.png', zoom=ZOOM)

In [ ]:
show_binned_residuals(sumatra_1[test_mask],
    [(pre_res, _BLUE, 'Pre-Kriging'), (post_res, _GREEN, 'Post-Kriging')],
    save='figs/ours-downsampled-gedi-field-binned-residuals.png')

In [ ]:
# Compare the 10 m and 100 m (downsampled) corrections against the same pre-kriging map.
post_res_10m = ours_post_pred_10m[test_mask] - sumatra_1[test_mask]
show_binned_residuals(sumatra_1[test_mask],
    [(pre_res, _BLUE, 'Pre-Kriging'), (post_res_10m, _AMBER, 'Post-Kriging (10m)'), (post_res, _GREEN, 'Post-Kriging (100m)')],
    save='figs/ours-both-gedi-field-binned-residuals.png')

In [ ]:
show_density(pre_pred, post_pred, sumatra_1, save='figs/ours-downsampled-gedi-field-density.png')
save_scatter_panels({"figs/scatter_ours_100m.png": (sumatra_1, post_pred)})

## ESA CCI prediction, post-kriging on GEDI

In [ ]:
# The ESA CCI map, corrected against GEDI.
CCI_MODEL = 'sumatra_cci_gedi'
post_path = paths.corrected_tif(ARCH, YEAR, ENS, CCI_MODEL, REFERENCE, STRIPE)

pre_pred, post_pred, sumatra_1 = load_pre_post_ref(CCI_INPUT, post_path)
pre_res, post_res, test_mask = print_metrics(pre_pred, post_pred, sumatra_1)

In [ ]:
r0, r1, c0, c1 = ZOOM
save_map_panels({
    "figs/esa_basemap.png":        pre_pred,
    "figs/esa_basemap-zoomed.png": pre_pred[r0:r1, c0:c1],
    "figs/esa_post.png":           post_pred,
    "figs/esa_post-zoomed.png":    post_pred[r0:r1, c0:c1],
})

In [ ]:
show_maps(pre_pred, post_pred, sumatra_1, save='figs/cci-gedi-field.png')
show_maps(pre_pred, post_pred, sumatra_1, save='figs/cci-gedi-field-zoom.png', zoom=ZOOM)

In [ ]:
show_binned_residuals(sumatra_1[test_mask],
    [(pre_res, _BLUE, 'Pre-Kriging'), (post_res, _GREEN, 'Post-Kriging')],
    save='figs/cci-gedi-field-binned-residuals.png', show_ylabel=False)

In [ ]:
show_density(pre_pred, post_pred, sumatra_1, save='figs/cci-gedi-field-density.png')
save_scatter_panels({
    "figs/scatter_esa_basemap.png": (sumatra_1, pre_pred),
    "figs/scatter_esa_post.png":    (sumatra_1, post_pred),
})

## Agreement of ours and CCI vs. GEDI (footprint-level)

Evaluate at the held-out GEDI **test footprints** (not per pixel). Both maps are compared at
100 m: **our downsampled** prediction and the **ESA CCI** map. Reads the per-run split indices
via `paths.split_pkl`, so it consumes exactly the splits each kriging run wrote (`FOOTPRINTS`
must match `--max_train_footprints`).

### Ours

In [ ]:
import geopandas as gpd
import pandas as pd

# Settings of the run being evaluated (must match kriging.sh).
FOOTPRINTS = 25000
MAX_DIFF, TEST_HO, VAL_HO = 15.0, 0.3, 0.2
path_gedi = paths.gedi_footprints()

# For the footprint-level comparison we use OUR DOWNSAMPLED (100 m) prediction — the strategy
# used against the ESA CCI map below — so both maps are compared like-for-like at the same
# resolution. Its split is a single file (the downsampled map is kriged as one raster).
OURS_MODEL = 'sumatra_downsampled_gedi_composite'


def footprint_agreement(pre_path, post_path, test_indices, path_gedi, resample_pre=False):
    """Pre/post RMSE, MAE, ME at the GEDI test footprints of `post_path`'s grid."""
    with rs.open(pre_path) as src1, rs.open(post_path) as src2:
        pre_pred = src1.read(1)
        post_pred = src2.read(1)
        target_crs = src1.crs
    gedi = gpd.read_file(path_gedi, engine='pyogrio').rename(columns={"index": "idx"}).to_crs(target_crs)
    gedi = gedi[gedi['idx'].isin(test_indices)]
    with rs.open(post_path) as src:
        width, height = src.width, src.height
        gedi[['row_idx', 'col_idx']] = gedi.apply(
            lambda r: src.index(r['geometry'].x, r['geometry'].y), axis=1).apply(pd.Series)
    gedi = gedi[(gedi['row_idx'] < height) & (gedi['row_idx'] >= 0) &
                (gedi['col_idx'] < width) & (gedi['col_idx'] >= 0)]
    gedi = gedi.groupby(['row_idx', 'col_idx'], as_index=False).agg(
        {'agbd': 'median', 'idx': list,
         **{c: 'first' for c in gedi.select_dtypes(include='number').columns
            if c not in ['row_idx', 'col_idx', 'agbd', 'idx']}})
    ys, xs = gedi['row_idx'].values.astype(int), gedi['col_idx'].values.astype(int)
    pre_agb, post_agb, ref_agb = pre_pred[ys, xs], post_pred[ys, xs], gedi['agbd'].values
    valid = (~np.isnan(pre_agb)) & (~np.isnan(post_agb)) & (~np.isnan(ref_agb))
    pre_res, post_res = pre_agb[valid] - ref_agb[valid], post_agb[valid] - ref_agb[valid]
    print(f"{'Metric':<10} {'Pre-Kriging':<15} {'Post-Kriging':<15}")
    print('-' * 40)
    print(f"{'RMSE':<10} {np.sqrt(np.mean(pre_res**2)):<15.2f} {np.sqrt(np.mean(post_res**2)):<15.2f}")
    print(f"{'MAE':<10} {np.mean(np.abs(pre_res)):<15.2f} {np.mean(np.abs(post_res)):<15.2f}")
    print(f"{'ME':<10} {np.mean(pre_res):<15.2f} {np.mean(post_res):<15.2f}")


# Return the flat list of test footprint indices from a split file. Handles both split
# formats: `test` may be a flat array of indices, or an array of per-pixel index lists.
def load_test_indices(split_path):
    with open(split_path, 'rb') as f:
        test = pickle.load(f)['test']
    out = []
    for entry in test:
        out.extend(np.atleast_1d(entry).tolist())
    return out


# Ours: single split file for the whole downsampled map (like CCI below).
ours_post = paths.corrected_tif(ARCH, YEAR, ENS, OURS_MODEL, REFERENCE, STRIPE)
ours_split = paths.split_pkl(OURS_MODEL, TEST_HO, VAL_HO, FOOTPRINTS, MAX_DIFF, STRIPE, REFERENCE)
test_indices = load_test_indices(ours_split)
print(f"Total test footprints: {len(test_indices)} | unique: {len(set(test_indices))}")
footprint_agreement(MERGED_INPUT, ours_post, test_indices, path_gedi)

### CCI

In [ ]:
# CCI: single split file for the whole map.
# CCI is an external (non-ensemble) map, so its model name carries no '_composite' suffix;
# the corrected raster and the split share this one name.
CCI_MODEL = 'sumatra_cci_gedi'
cci_post = paths.corrected_tif(ARCH, YEAR, ENS, CCI_MODEL, REFERENCE, STRIPE)
cci_split = paths.split_pkl(CCI_MODEL, TEST_HO, VAL_HO, FOOTPRINTS, MAX_DIFF, STRIPE, REFERENCE)
test_indices = load_test_indices(cci_split)
print(f"Total test footprints: {len(test_indices)} | unique: {len(set(test_indices))}")
footprint_agreement(CCI_INPUT, cci_post, test_indices, path_gedi)